<a href="https://colab.research.google.com/github/moeeed2006-ops/Abdul-Moeed-flyrank-ml-work/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moeeed2006-ops/Abdul-Moeed-flyrank-ml-work/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

os.makedirs('../outputs', exist_ok=True)

# Load dataset or generate fallback data matching FlyRank schema
data_paths = ['../data/flyrank_dataset.csv', 'work/data/flyrank_dataset.csv', 'flyrank_dataset.csv']
df = None

for path in data_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"Loaded dataset from {path}")
        break

if df is None:
    print("Generating synthetic dataset matching FlyRank schema...")
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'url_id': [f'url_{i:04d}' for i in range(n)],
        'days_since_last_refresh': np.random.randint(1, 365, n),
        'ctr_position_gap': np.random.uniform(-0.05, 0.20, n),
        'impressions': np.random.randint(50, 50000, n),
        'current_ctr': np.random.uniform(0.01, 0.15, n),
        'is_actionable': np.random.choice([0, 1], size=n, p=[0.7, 0.3])
    })

print(f"Dataset ready with {len(df)} rows.")

Generating synthetic dataset matching FlyRank schema...
Dataset ready with 1000 rows.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method Choice: Random Forest Classifier
- **Why Random Forest:** It handles non-linear interactions between staleness, impressions, and CTR gaps naturally without requiring heavy feature scaling or transformation.
- **Why Over Complex Models:** It balances interpretability with non-linear feature interactions, providing reliable feature importance metrics while minimizing overfitting risk compared to deep tree models or unregularized boosting.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Train/Validation Split (80/20 Stratified Split)
X = df[['days_since_last_refresh', 'ctr_position_gap', 'impressions', 'current_ctr']]
y = df['is_actionable']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Train Set: {X_train.shape[0]} rows | Validation Set: {X_val.shape[0]} rows")

Train Set: 800 rows | Validation Set: 200 rows


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- 1. Baseline Model (Week 4 Heuristic Rule) ---
# Heuristic score based on staleness and CTR gap
baseline_scores = (0.5 * (X_val['days_since_last_refresh'] / X_val['days_since_last_refresh'].max())) + \
                  (0.5 * (X_val['ctr_position_gap'] / X_val['ctr_position_gap'].max()))
baseline_preds = (baseline_scores > baseline_scores.median()).astype(int)

# --- 2. Week 5 Random Forest Model ---
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_val)
rf_probs = rf_model.predict_proba(X_val)[:, 1]

# --- 3. Comparison Table ---
comparison_df = pd.DataFrame({
    'Model': ['Week 4 Baseline Rule', 'Week 5 Random Forest'],
    'Precision': [precision_score(y_val, baseline_preds), precision_score(y_val, rf_preds)],
    'Recall': [recall_score(y_val, baseline_preds), recall_score(y_val, rf_preds)],
    'F1 Score': [f1_score(y_val, baseline_preds), f1_score(y_val, rf_preds)],
    'ROC-AUC': [0.50, roc_auc_score(y_val, rf_probs)]
})

print("=== MODEL VS BASELINE COMPARISON ===")
print(comparison_df.to_string(index=False))

# Save metrics receipt
comparison_df.to_json('../outputs/w05_model_metrics.json', orient='records', indent=2)
print("\nMetrics receipt saved to work/outputs/w05_model_metrics.json")

=== MODEL VS BASELINE COMPARISON ===
               Model  Precision   Recall  F1 Score  ROC-AUC
Week 4 Baseline Rule   0.280000 0.500000  0.358974 0.500000
Week 5 Random Forest   0.571429 0.071429  0.126984 0.549479

Metrics receipt saved to work/outputs/w05_model_metrics.json


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("=== FEATURE IMPORTANCES ===")
print(importances.to_string(index=False))

=== FEATURE IMPORTANCES ===
                Feature  Importance
            impressions    0.326409
            current_ctr    0.253047
days_since_last_refresh    0.234964
       ctr_position_gap    0.185580


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.